# Shortcuts in self-supervised tasks

A pretext task is useless if the network can solve it with **low-level cues** instead of semantic
understanding. This notebook illustrates two such cues and the usual fixes:

1. **Chromatic aberration**: lenses focus colours slightly differently, so colour fringes reveal
   where a patch was in the image (a shortcut for patch-position tasks such as jigsaw or context
   prediction). Fix: grayscale or color dropping.
2. **Colour histograms**: patches of the same image share very similar intensity histograms, a
   shortcut for contrastive methods matching two crops (noted in SimCLR). Fix: colour jittering.

In [ ]:
import os
import sys

if "google.colab" in sys.modules:
    !git clone -q https://github.com/ckurtz/cs294-158-ssl /content/cs294-158-ssl
    %cd /content/cs294-158-ssl

import matplotlib.pyplot as plt
import numpy as np
from PIL import Image
from torchvision import transforms

%matplotlib inline
rng = np.random.default_rng(0)

In [ ]:
def color_dropping(image, keep_channel=0, rng=None):
    """
    Keep one colour channel and replace the others with Gaussian noise.

    Parameters
    ----------
    image : PIL.Image.Image
        Input image.
    keep_channel : int, default=0
        Index of the channel kept (0=R, 1=G, 2=B).
    rng : numpy.random.Generator, optional
        Random generator.

    Returns
    -------
    PIL.Image.Image
        RGB image with two noisy channels.
    """
    rng = rng or np.random.default_rng()
    arr = np.asarray(image.convert("RGB")).astype(np.float32)
    out = arr.copy()
    for c in range(3):
        if c != keep_channel:
            out[..., c] = rng.normal(arr[..., c].mean(), arr[..., keep_channel].std(), arr.shape[:2])
    return Image.fromarray(np.clip(out, 0, 255).astype(np.uint8))


def to_grayscale(image):
    """
    Convert an image to 3-channel grayscale (ITU-R 601 luma).

    Parameters
    ----------
    image : PIL.Image.Image
        Input image.

    Returns
    -------
    PIL.Image.Image
        Grayscale image repeated on 3 channels.
    """
    return image.convert("L").convert("RGB")


def random_patches(image, n_patches=4, patch_size=128, rng=None):
    """
    Crop random square patches from an image.

    Parameters
    ----------
    image : PIL.Image.Image
        Input image.
    n_patches : int, default=4
        Number of patches.
    patch_size : int, default=128
        Patch side in pixels.
    rng : numpy.random.Generator, optional
        Random generator.

    Returns
    -------
    list of numpy.ndarray
        Patches of shape ``(patch_size, patch_size, 3)``.
    """
    rng = rng or np.random.default_rng()
    arr = np.asarray(image.convert("RGB"))
    h, w, _ = arr.shape
    tops = rng.integers(0, h - patch_size, n_patches)
    lefts = rng.integers(0, w - patch_size, n_patches)
    return [arr[t:t + patch_size, l:l + patch_size] for t, l in zip(tops, lefts)]


def show_images(images, titles, size=4):
    """
    Display images side by side.

    Parameters
    ----------
    images : list of PIL.Image.Image
        Images to display.
    titles : list of str
        One title per image.
    size : int, default=4
        Width of each subplot in inches.

    Returns
    -------
    None
    """
    fig, axs = plt.subplots(1, len(images), figsize=(size * len(images), size))
    for ax, img, title in zip(np.atleast_1d(axs), images, titles):
        ax.imshow(img)
        ax.set_title(title)
        ax.axis("off")
    plt.show()


def plot_patch_histograms(patch_sets, row_titles, bins=50):
    """
    Plot per-channel intensity histograms of several patches.

    Parameters
    ----------
    patch_sets : list of list of numpy.ndarray
        One list of patches per row.
    row_titles : list of str
        One title per row.
    bins : int, default=50
        Number of histogram bins.

    Returns
    -------
    None
    """
    n_rows, n_cols = len(patch_sets), len(patch_sets[0])
    fig, axs = plt.subplots(n_rows, n_cols, figsize=(4 * n_cols, 3 * n_rows), squeeze=False)
    for r, (patches, title) in enumerate(zip(patch_sets, row_titles)):
        for c, patch in enumerate(patches):
            for ch, color in enumerate(("red", "green", "blue")):
                axs[r, c].hist(patch[..., ch].ravel(), bins=bins, range=(0, 255),
                               color=color, alpha=0.4)
            axs[r, c].set_title(f"{title} - patch {c + 1}")
    plt.tight_layout()
    plt.show()

## 1. Chromatic aberration
Look at the purple fringes around the branches; grayscale and color dropping remove the cue.

In [ ]:
image = Image.open("sample_images/chrom_ab_demo.png").convert("RGB")
show_images([image, color_dropping(image, keep_channel=0, rng=rng), to_grayscale(image)],
            ["Original", "Color dropping (keep R)", "Grayscale"], size=5)

## 2. Colour histograms of patches
The same random patch positions are used before and after jittering, so the rows are directly comparable.

In [ ]:
image = Image.open("sample_images/n01537544_19414.JPEG").convert("RGB")
jittered = transforms.ColorJitter(0.4, 0.4, 0.4, 0.1)(image)
show_images([image, jittered], ["Original", "Colour jittered"], size=5)

seed = 1
plot_patch_histograms(
    [random_patches(image, rng=np.random.default_rng(seed)),
     random_patches(jittered, rng=np.random.default_rng(seed))],
    ["Original", "Jittered"],
)

In SimCLR, jitter is applied **independently to each view**, so two crops of the same image no
longer share their colour distribution and the network has to rely on content instead.

In [ ]:
views = [transforms.ColorJitter(0.4, 0.4, 0.4, 0.1)(image) for _ in range(2)]
plot_patch_histograms(
    [random_patches(v, n_patches=2, rng=np.random.default_rng(seed)) for v in views],
    ["View 1", "View 2"],
)